# P01, P12–P14 — Demostración con datos SIMULADOS
## (a) Paso del protocolo y objetivo
**Todo lo de este cuaderno es «Simulado (demostración)»: no son hallazgos de campo y nada se suma a las cifras reales.**
- **P01** (diagnóstico, **OE1**): línea base de tiempos **simulada** (120 respuestas inventadas del diseño V1.2).
- **P12–P13** (piloto exploratorio de 60 sesiones asistidas; **OE3 y OE4**): hoy **planificado**; sin sesiones reales. La demostración usa un registro de 60 sesiones **inventadas**.
- **P14** (análisis estadístico): demuestra que el *pipeline* corre (normalidad de Shapiro-Wilk → t pareada o Wilcoxon, tamaño del efecto, alfa de Cronbach).
- La **demostración completa** (`demostracion_simulada.py`, 5 de octubre) encadena ingesta → partición → evaluación → congelamiento → sesiones → tablero.

## (b) Código del repositorio que lo implementa
`demostracion_simulada.py`, `deteccion_simulado.py` (impide que lo simulado cuente como real), `simular_P14_n120.py`, `simular_analisis_p14.py`, `stats_analysis.py` y `analizar_piloto.py` (**modo `--demo-simulada` solo**). Entradas: `Lote1_Transcripcion_SIMULADO_v2.xlsx` y `Registro_Sesiones_Piloto_SIMULADO_v4.xlsx`. Salidas: `evidencias/simulado_demostracion/<ejecución>/`.

In [1]:
# ----- Preparación mínima (igual en todos los cuadernos) -----
import os, sys, zipfile
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CARPETA_DRIVE = Path("/content/drive/MyDrive/MPSR_colab")
    BASE = Path("/content/mpsr")
else:
    CARPETA_DRIVE = Path(os.environ.get("MPSR_DRIVE", ".")).resolve()
    BASE = Path(os.environ.get("MPSR_BASE", "mpsr_trabajo")).resolve()
BASE.mkdir(parents=True, exist_ok=True)
ZIP_PUBLICO = CARPETA_DRIVE / "MPSR_colab_publico.zip"
if not ZIP_PUBLICO.exists():
    raise SystemExit(f"Falta {ZIP_PUBLICO}. Sube MPSR_colab_publico.zip a la carpeta MPSR_colab de tu Drive (ver LEEME_colab_por_paso.md).")
if not (BASE / "scripts" / "colab_util.py").exists():          # solo descomprime la primera vez
    with zipfile.ZipFile(ZIP_PUBLICO) as z:
        z.extractall(BASE)
sys.path.insert(0, str(BASE / "scripts"))
import colab_util as U
U.iniciar(BASE, CARPETA_DRIVE)                                  # rutas, paquete privado (si existe) y detección de Rasa
from colab_util import *                                        # rotulo, comparar, sha256, leer_json, tabla_scripts, ic_bootstrap, ...

Listo · BASE = base · paquete privado: SÍ · Rasa: no instalado (no hace falta)


In [2]:
tabla_scripts(scripts_de("13"))

▌ORIGEN: Código del proyecto (no es dato del estudio)


,script,paso del protocolo,propósito (docstring),entradas,salidas,cómo ejecutarlo,usa Rasa,rol en este cuaderno
0,analizar_piloto.py,"P12–P14 (piloto), usa el registro de sesiones","Análisis del piloto exploratorio (protocolo V1.3: una sesión asistida, n = 60 sesiones elegibles).",Registro de sesiones (.xlsx) elegibles,"Análisis del piloto sobre sesiones elegibles: tiempos, alfa de Cronbach, prueba final del modelo (UNA vez)",python scripts/analizar_piloto.py --registro docs/piloto/privado/Registro_lleno.xlsx --modelo-congelado logs/v3_real/modelo_congelado.json \,No,Documentado — NO ejecutado sobre datos reales (gastaría la prueba final única)
1,demostracion_simulada.py,2.14 (demostración),"Demostración SIMULADA completa del flujo del curso (protocolo 2.14, «Datos simulados»).",Lote1_Transcripcion_SIMULADO_v2.xlsx y Registro_Sesiones_Piloto_SIMULADO_v4.xlsx,evidencias/simulado_demostracion/<ejecución>/ (6 etapas + informe),python scripts/demostracion_simulada.py --demo-simulada [--ejecucion <nombre>],No,"Resultado guardado (etapa 15, Simulado)"
2,deteccion_simulado.py,2.14 (datos simulados),"Detección común de libros y tablas SIMULADOS o SINTÉTICOS (la usan analizar_piloto.py, conciliar_seguimiento.py e ingest_real_lote.py).",Libros .xlsx y tablas,Decisión «¿es simulado/sintético?»; bloquea que lo simulado cuente como real,python scripts/deteccion_simulado.py --help,No,Documentado (impide mezclar rótulos)
3,simular_P14_n120.py,P14 (ilustrativo),"*** SIMULACIÓN ILUSTRATIVA -- NO SON DATOS DE CAMPO REALES (salvo la línea base) *** Usa la línea base REAL-SIMULADA de P01 (Encuestas_simuladas_TramiFacil_MPSR_120_v2.xlsx, n=120, generada por el propio tesist",Línea base simulada P01 (n = 120),Resultados simulados de P14,python scripts/simular_P14_n120.py --help,No,Documentado (Simulado)
4,simular_analisis_p14.py,P14 (ilustrativo),*** SIMULACIÓN ILUSTRATIVA -- NO SON DATOS DE CAMPO REALES *** Este script demuestra que el pipeline de análisis estadístico del protocolo (P14: Shapiro-Wilk -> t de Student pareada / Wilcoxon) funciona correct,Datos inventados,Demuestra que el pipeline estadístico corre,python scripts/simular_analisis_p14.py --help,No,Documentado (Simulado)
5,stats_analysis.py,P14,"P14 — Análisis estadístico (OE2, OE3, OE4).",Datos de línea base y de sesiones,"Pruebas de normalidad, t pareada o Wilcoxon, tamaño del efecto",python scripts/stats_analysis.py modelos,No,Documentado


## (c) Celdas de código
### 1. P01 — Línea base de tiempos SIMULADA

In [3]:
from openpyxl import load_workbook
wbq = load_workbook(BASE / "corpus/Encuestas_simuladas_TramiFacil_MPSR_120_v2.xlsx", read_only=True, data_only=True)
filas_q = list(wbq.worksheets[0].iter_rows(values_only=True))
wbq.close()
cab = [str(c) for c in filas_q[0]]
q = pd.DataFrame(filas_q[1:], columns=cab)
col_t = next(c for c in cab if c.startswith("Tiempo presencial"))
rotulo(ORIGEN_SIMULADO, "línea base inventada del diseño V1.2 (n = 120): NO es recolección de campo")
print("Respuestas simuladas:", len(q))
display(q[col_t].value_counts().rename("respuestas simuladas").to_frame())

▌ORIGEN: Simulado (demostración) — línea base inventada del diseño V1.2 (n = 120): NO es recolección de campo
Respuestas simuladas: 120


,respuestas simuladas
Tiempo presencial,
30–60 min,40
1–2 horas,28
15–30 min,16
2–4 horas,14
Menos de 15 min,11
Más de 4 horas / varios días,11


### 2. P14 — Análisis estadístico sobre datos simulados (resultado guardado)

In [4]:
for archivo in ("SIMULACION_resultado_P14_n120.json",):
    sim14 = leer_json("logs/simulaciones_P14/" + archivo)
    rotulo(ORIGEN_SIMULADO, "resultado guardado de simular_P14_n120.py — " + sim14["ADVERTENCIA"][:150] + "…")
    t = sim14["tiempo_atencion"]
    display(pd.DataFrame([{"variable": t["variable"], "n": t["n"], "media pre": t["media_pre"], "media post (simulada)": t["media_post"], "prueba": t["prueba_usada"], "p": t["p_value"], "decisión H0": t["decision_H0"]}]))
    print("Ninguna de estas cifras debe reportarse como hallazgo real de la tesis.")

▌ORIGEN: Simulado (demostración) — resultado guardado de simular_P14_n120.py — La línea base (pre) proviene del archivo Encuestas_simuladas_TramiFacil_MPSR_120_v2.xlsx, generado por el tesista para esta demostración del curso (no…


,variable,n,media pre,media post (simulada),prueba,p,decisión H0
0,Tiempo de atención (minutos),119,88.97,33.26,Wilcoxon (rangos con signo),2.880919e-21,Se rechaza H0 (diferencia significativa)


Ninguna de estas cifras debe reportarse como hallazgo real de la tesis.


### 3. Demostración completa de punta a punta
La celda siguiente necesita el tablero simulado.

In [5]:
ts = leer_json("evidencias/simulado_demostracion/20261005_demostracion/06_tablero/estado_compuertas_SIMULADO.json")

### Demostración del flujo completo con datos SIMULADOS
**Qué se hace:** se recorre de punta a punta la demostración simulada (`demostracion_simulada.py`, ejecutada el 5 de octubre de 2026): ingesta → partición → evaluación → congelamiento → sesiones → tablero. Todo con libros inventados: `Lote1_Transcripcion_SIMULADO_v2.xlsx` y `Registro_Sesiones_Piloto_SIMULADO_v4.xlsx`.
**Por qué:** muestra que el *procedimiento* funciona (y que el código detecta lo que debe) sin gastar ni tocar datos reales. **Nada de esto es un hallazgo.**
**Rótulo:** todo lo de esta etapa es **Simulado (demostración)** y nunca se suma a las cifras reales.

In [6]:
DEMO = BASE / "evidencias/simulado_demostracion/20261005_demostracion"
rotulo(ORIGEN_SIMULADO, "1. Ingesta (libro de transcripción simulado)")
lf = pd.read_csv(DEMO / "01_ingesta/lote1_real_final_SIMULADO.csv", dtype=str, keep_default_na=False, encoding="utf-8")
print("Frases del lote simulado tras la revisión automática:", len(lf), "| intenciones:", lf["intent"].nunique() if "intent" in lf else "—")

rotulo(ORIGEN_SIMULADO, "2. Partición v3 con el lote simulado")
sp = pd.read_csv(DEMO / "02_particion/dataset_split_v3_SIMULADO.csv", dtype=str, keep_default_na=False, encoding="utf-8")
print("Partición:", sp["split"].value_counts().to_dict())
fuga = int((sp[sp["split"].isin(["train", "validation", "test"])].groupby("base_phrase_id")["split"].nunique() > 1).sum())
print("Grupos de paráfrasis en más de una partición:", fuga)

rotulo(ORIGEN_SIMULADO, "3. Evaluación (mejor configuración ya elegida; una evaluación del test)")
er = json.loads((DEMO / "03_evaluacion/eval_real_resumen_SIMULADO.json").read_text(encoding="utf-8"))
display(pd.DataFrame([{"método": m, "F1 macro (promedio de semillas)": round(v["f1_macro"][0], 4), "IC inferior": round(v["f1_macro"][1], 4), "IC superior": round(v["f1_macro"][2], 4)} for m, v in er["metodos"].items()]))

rotulo(ORIGEN_SIMULADO, "4. «Congelamiento» de demostración (un marcador con otro nombre, no el modelo real)")
fzs = json.loads((DEMO / "04_congelado/modelo_congelado_SIMULADO.json").read_text(encoding="utf-8"))
mod_sim = DEMO / "04_congelado/modelo_demostracion_SIMULADO.tar.gz"
print("Estado declarado:", fzs["estado"])
print("sha256 del marcador coincide con el congelamiento simulado:", sha256(mod_sim) == fzs["sha256"]["modelo"])

rotulo(ORIGEN_SIMULADO, "5. Sesiones simuladas (60) y alfa de Cronbach")
an = json.loads((DEMO / "05_sesiones/analisis_piloto_SIMULADO.json").read_text(encoding="utf-8"))
print("Sesiones simuladas elegibles:", an["n"], "| alfa guardado de la demostración:", round(an["satisfaccion"]["alfa_cronbach"], 4))
from openpyxl import load_workbook
wbs = load_workbook(BASE / "docs/piloto/ejemplos_simulados/Registro_Sesiones_Piloto_SIMULADO_v4.xlsx", read_only=True, data_only=True)
filas_s = list(wbs["Sesiones"].iter_rows(values_only=True))
wbs.close()
enc_s = next(i for i, r in enumerate(filas_s[:10]) if r and str(r[0]).startswith("C") and "sesi" in str(r[0]))
cab = [str(c) if c else "" for c in filas_s[enc_s]]
j_el = cab.index("Elegible y completa")
j_it = [next(j for j, c in enumerate(cab) if c.startswith(f"Post ítem {i}:")) for i in range(1, 9)]
datos_s = [r for r in filas_s[enc_s + 1:] if r and r[0] and re.match(r"^(SA|PP)\d", str(r[0])) and str(r[j_el]) == "Sí"]
Xs = np.array([[float(r[j]) for j in j_it] for r in datos_s])
a_sim, _, _ = alfa_cronbach(Xs)
print(f"Alfa recalculado desde el registro simulado: {a_sim:.4f} (n = {len(Xs)}); la demostración NO es un resultado del pre-piloto ni del piloto.")
comparar("Simulado", "alfa de la demostración simulada (Simulado)", a_sim, an["satisfaccion"]["alfa_cronbach"], "evidencias/simulado_demostracion/…/05_sesiones", tol=1e-6)

rotulo(ORIGEN_SIMULADO, "6. Tablero de la demostración")
print("Compuertas «Cumplida (Simulado)» en la demostración:", [c["id"] for c in ts["compuertas"] if "Simulado" in c["estado"]], "| ninguna cuenta como real")
inf = (DEMO / "INFORME_DEMOSTRACION_SIMULADA.md").read_text(encoding="utf-8")
print("\nAdvertencias del informe de la demostración (resumen):")
for linea in re.findall(r"^\d\. \*\*.*$", inf, re.M)[:4]:
    print(" -", linea[:260])
mostrar_comparaciones(etapas=["Simulado"], titulo="Esta sección — recalculado frente a reportado (Simulado)")

▌ORIGEN: Simulado (demostración) — 1. Ingesta (libro de transcripción simulado)
Frases del lote simulado tras la revisión automática: 261 | intenciones: 54
▌ORIGEN: Simulado (demostración) — 2. Partición v3 con el lote simulado
Partición: {'train': 708, 'test': 151, 'validation': 110}
Grupos de paráfrasis en más de una partición: 0
▌ORIGEN: Simulado (demostración) — 3. Evaluación (mejor configuración ya elegida; una evaluación del test)


,método,F1 macro (promedio de semillas),IC inferior,IC superior
0,rasa,0.8206,0.7318,0.8452
1,svm,0.8094,0.7122,0.8507


▌ORIGEN: Simulado (demostración) — 4. «Congelamiento» de demostración (un marcador con otro nombre, no el modelo real)
Estado declarado: SIMULADO — modelo de demostración; NO es el modelo congelado del piloto
sha256 del marcador coincide con el congelamiento simulado: True
▌ORIGEN: Simulado (demostración) — 5. Sesiones simuladas (60) y alfa de Cronbach
Sesiones simuladas elegibles: 60 | alfa guardado de la demostración: 0.5488
Alfa recalculado desde el registro simulado: 0.5488 (n = 60); la demostración NO es un resultado del pre-piloto ni del piloto.
▌ORIGEN: Simulado (demostración) — 6. Tablero de la demostración
Compuertas «Cumplida (Simulado)» en la demostración: ['G1', 'G2', 'G3', 'G5', 'G7'] | ninguna cuenta como real

Advertencias del informe de la demostración (resumen):
 - 1. **Las frases del lote son generadas y no miden lenguaje real.** Las escribió quien preparó el libro de prueba (varias son idénticas a frases del corpus sintético); las cifras de las etapas 2 y 3 no dicen 

,paso,cifra,recalculado,reportado,fuente de lo reportado,coincide
0,Simulado,alfa de la demostración simulada (Simulado),0.548783,0.548783,evidencias/simulado_demostracion/…/05_sesiones,Sí


### Cómo leer el resultado de esta sección
- La demostración **no mide nada del estudio**: las frases las escribió quien armó el libro de prueba (varias son idénticas al corpus sintético), las «consultas» de las sesiones son el texto de las tarjetas, y la prueba final usa un **predictor simulado**.
- Sirve para comprobar que cada eslabón (ingesta → partición → evaluación → congelamiento → sesiones → tablero) corre y deja huellas. El alfa de la demostración (≈ 0,55) **no** tiene relación con el 0,366 del pre-piloto real.

In [7]:
cierre("P01 y P12–P14 (Simulado)")

P01 y P12–P14 (Simulado) — recalculado frente a reportado: 1 cifras · coinciden: 1 · NO coinciden: 0


,paso,cifra,recalculado,reportado,fuente de lo reportado,coincide
0,Simulado,alfa de la demostración simulada (Simulado),0.548783,0.548783,evidencias/simulado_demostracion/…/05_sesiones,Sí


DISCREPANCIAS (recalculado ≠ reportado): ninguna
Rasa disponible: False · paquete privado: True


## (d) Cómo leer el resultado
- La línea base simulada es una tabla de rangos de tiempo inventados; la reducción de tiempo de P14 es **ilustrativa** (ADVERTENCIA del propio archivo). Sirve para probar el *pipeline*, no para afirmar nada sobre la MPSR.
- En la demostración completa, G7 aparece «Cumplida (Simulado)» porque se inventaron 60 sesiones; **no cuenta** para el tablero real (5 de 7, con G7 planificada). El alfa de la demostración (≈ 0,55) no tiene relación con el 0,366 del pre-piloto real.

## Limitaciones
- Las frases del lote simulado las escribió quien armó el libro de prueba (varias son idénticas al corpus sintético) y las «consultas» de las sesiones son el texto de las tarjetas; la prueba final usa un **predictor simulado**.
- Los resultados de este cuaderno nunca deben citarse como evidencia empírica de la tesis: solo como prueba de que el procedimiento se puede ejecutar.